In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import graphviz
import torch

In [2]:
STORAGE_PATH = './figures/'
os.makedirs(STORAGE_PATH, exist_ok = True)

In [3]:
# Simple local sentiment (SST-2)
#   16, 18, 143, 224
# Long-range reversal (IMDb)
#   176, 273, 872
# Long-range reversal (IMDb-1k)
#   11, 279
# Keyword trap (IMDb/IMDb-1k + SST-2)
#   SST-2 : 92, 338
#   IMDb: 176, 273
#   IMDb-1k: 11, 178
# Long noisy reviews (IMDb)
#   176, 384, 411, 872
# Long noisy reviews (IMDb-1k)
#   11, 13, 135, 143
# Misclassified samples (IMDb/IMDb-1k + SST-2)
#   SST-2 : 17, 111
#   IMDb : 176?, 273?
#   IMDb-1k : 1, 143

# SST-2: [16, 17, 18, 92, 111, 143, 224, 338] # Extra: 20
# IMDb-1k: [1, 11, 13, 135, 143, 178, 279]

In [3]:
def create_grey_to_black_colormap():
  color_dictionary = {
    'red': [
      (0.0, 0.75, 0.75),
      (1.0, 0.0, 0.0)
    ],
    'green': [
      (0.0, 0.75, 0.75),
      (1.0, 0.0, 0.0)
    ],
    'blue': [
      (0.0, 0.75, 0.75),
      (1.0, 0.0, 0.0)
    ]
  }
    
  return mcolors.LinearSegmentedColormap('GreyToBlack', color_dictionary)

def create_yellow_colormap():
  light_yellow = mcolors.hex2color('#FFF9E2')
  dark_yellow = mcolors.hex2color('#FFD500')
    
  color_dictionary = {
    'red':   [(0.0, light_yellow[0], light_yellow[0]), (1.0, dark_yellow[0], dark_yellow[0])],
    'green': [(0.0, light_yellow[1], light_yellow[1]), (1.0, dark_yellow[1], dark_yellow[1])],
    'blue':  [(0.0, light_yellow[2], light_yellow[2]), (1.0, dark_yellow[2], dark_yellow[2])]
  }

  return mcolors.LinearSegmentedColormap('YellowGradient', color_dictionary)

def create_color_gradient(values, colormap):
  norm = plt.Normalize(np.min(values), np.max(values))
  normalized_values = norm(values)    
  rgb_colors = colormap(normalized_values)
  return [mcolors.to_hex(color) for color in rgb_colors]

In [4]:
grey_to_black_cmap = create_grey_to_black_colormap()
light_to_dark_yellow_cmap = create_yellow_colormap()

In [5]:
def visualize_graph(path, method, filename, edge):
  graph = torch.load(os.path.join(path, method, filename))
  graph_df = pd.DataFrame({
    'from_index' : graph['edge_index'][0],
    'from_token' : graph['from_tokens'],
    'to_index' : graph['edge_index'][1],
    'to_token' : graph['to_tokens'],
  })
  for dimension in range(graph['edge_attr'].size(1)):
    graph_df[f'edge_weight_{dimension}'] = graph['edge_attr'][:, dimension].tolist()
  
  weight_columns = [f'edge_weight_{dimension}'for dimension in range(graph['edge_attr'].size(1))]
  incoming = graph_df[['to_index', 'to_token'] + weight_columns].rename(columns = {'to_index' : 'id', 'to_token' : 'token'}).groupby(['id', 'token'])[weight_columns].sum()
  outgoing = graph_df[['from_index', 'from_token'] + weight_columns].rename(columns = {'from_index' : 'id', 'from_token' : 'token'}).groupby(['id', 'token'])[weight_columns].sum()
  nodes = incoming.add(outgoing, fill_value = 0).reset_index()
  
  edge_colors = create_color_gradient(graph_df[f'edge_weight_{edge}'], grey_to_black_cmap)
  node_colors = create_color_gradient(nodes[f'edge_weight_{edge}'], light_to_dark_yellow_cmap)

  graph = graphviz.Digraph(f'{filename.replace(".pt", "")}-{edge}', engine = 'dot') # engine = 'circo', 'twopi', 'dot
  graph.graph_attr['dpi'] = '300'

  min_size = 0.5
  max_size = 2.5
  min_font = 12
  max_font = 72
  min_weight = nodes[f'edge_weight_{edge}'].min()
  max_weight = nodes[f'edge_weight_{edge}'].max()
  for i, node in nodes.iterrows():
    normalized = (node[f'edge_weight_{edge}'] - min_weight) / (max_weight - min_weight) if max_weight != min_weight else 0.5
    size = min_size + normalized * (max_size - min_size)
    fontsize = min_font + normalized * (max_font - min_font)
    graph.node(str(node['id']), label = node['token'], color = '#363636', style = 'filled', fillcolor = node_colors[i], shape = 'oval', width = str(1.85 * size), height = str(size), fixedsize = 'true', fontsize = str(fontsize))

  for i, row in graph_df.iterrows():
    graph.edge(str(row['from_index']), str(row['to_index']), arrowsize = '0.5', color = edge_colors[i])
  os.makedirs(os.path.join(STORAGE_PATH, method), exist_ok = True)
  graph.render(directory = os.path.join(STORAGE_PATH, method, f'{filename.replace(".pt", "")}-{edge}'), format = 'png', view = True)

In [ ]:
for i in [16, 17, 18, 92, 143, 224, 338]:
  visualize_graph(path = './graphs/', method = 'sliding_windows', filename = f'SST-2-test-{i}.pt', edge = 0)
# for i in [1, 11, 13, 135, 143, 178, 279]:
#   visualize_graph(path = './graphs/', method = 'sliding_windows', filename = f'IMDb-1k-test-{i}.pt', edge = 0)

In [ ]:
for i in [16, 17, 18, 92, 143, 224, 338]:
  visualize_graph(path = './graphs/', method = 'attention_distillation', filename = f'SST-2-test-{i}.pt', edge = 0)
# for i in [1, 11, 13, 135, 143, 178, 279]:
#   visualize_graph(path = './graphs/', method = 'attention_distillation', filename = f'IMDb-1k-test-{i}.pt', edge = 0) # There are L edge features, where L is the count of models layer in the PLM (BERT-base)

dot: graph is too large for cairo-renderer bitmaps. Scaling by 0.800992 to fit
dot: graph is too large for cairo-renderer bitmaps. Scaling by 0.860772 to fit


In [5]:
def visualize_graph(path, method, filename, edge):
  graph = torch.load(os.path.join(path, method, filename))
  graph_df = pd.DataFrame({
    'from_index' : graph['edge_index'][0],
    'from_token' : [graph['tokens'][i] for i in graph['edge_index'][0]],
    'to_index' : graph['edge_index'][1],
    'to_token' : [graph['tokens'][i] for i in graph['edge_index'][1]],
  })
  for dimension in range(graph['edge_attr'].size(1)):
    graph_df[f'edge_weight_{dimension}'] = graph['edge_attr'][:, dimension].tolist()
  
  # Ignore 0.0 edges
  graph_df = graph_df[graph_df[f'edge_weight_{edge}'] > 0.0]

  weight_columns = [f'edge_weight_{dimension}'for dimension in range(graph['edge_attr'].size(1))]
  incoming = graph_df[['to_index', 'to_token'] + weight_columns].rename(columns = {'to_index' : 'id', 'to_token' : 'token'}).groupby(['id', 'token'])[weight_columns].sum()
  outgoing = graph_df[['from_index', 'from_token'] + weight_columns].rename(columns = {'from_index' : 'id', 'from_token' : 'token'}).groupby(['id', 'token'])[weight_columns].sum()
  nodes = incoming.add(outgoing, fill_value = 0).reset_index()

  pattern_D = r'\[D\]'
  pattern_T = r'\[T-\d+\]'
  mask = (
    # Exclude edges connected to [D]
    ~graph_df['from_token'].str.contains(pattern_D) &
    ~graph_df['to_token'].str.contains(pattern_D) &
    # Exclude self-loops of [T-i]
    ~((graph_df['from_token'] == graph_df['to_token']) & graph_df['from_token'].str.contains(pattern_T)) &
    # Only include edges connected to [T-i] nodes
    (graph_df['from_token'].str.contains(pattern_T) | graph_df['to_token'].str.contains(pattern_T))
  )
  edge_colors = pd.Series('#cccccc', index=graph_df.index)  # default grey for excluded edges
  edge_colors[mask] = create_color_gradient(graph_df.loc[mask, f'edge_weight_{edge}'], grey_to_black_cmap) # ALTER edge_weight_0
  node_mask = ~(nodes['token'].str.contains(pattern_D) | nodes['token'].str.contains(pattern_T))
  node_colors = pd.Series('#cccccc', index = nodes.index)  # default grey for excluded nodes
  node_colors[node_mask] = create_color_gradient(nodes.loc[node_mask, f'edge_weight_{edge}'], light_to_dark_yellow_cmap)

  graph = graphviz.Digraph(f'{filename.replace(".pt", "")}-{edge}', engine = 'dot') # engine = 'circo', 'twopi', 'dot
  graph.graph_attr['dpi'] = '300'

  with graph.subgraph() as highest_rank:
    highest_rank.attr(rank='same')
    for i, node in nodes[nodes['token'].str.contains(r'\[D\]')].iterrows():
      highest_rank.node(str(node['id']), label=node['token'], color='#363636', style='filled', fillcolor = node_colors[i], shape='oval')

  with graph.subgraph() as second_rank:
    second_rank.attr(rank='same')
    for i, node in nodes[nodes['token'].str.contains(r'\[T-\d+\]')].iterrows():
      second_rank.node(str(node['id']), label=node['token'], color='#363636', style='filled', fillcolor = node_colors[i], shape='oval')

  min_size = 0.5
  max_size = 2.5
  min_font = 12
  max_font = 72
  min_weight = nodes[f'edge_weight_{edge}'].min()
  max_weight = nodes[f'edge_weight_{edge}'].max()
  with graph.subgraph() as lowest_rank:
    lowest_rank.attr(rank='same')
    for i, node in nodes[~nodes['token'].str.contains(r'\[D\]|\[T-\d+\]')].iterrows():
      normalized = (node[f'edge_weight_{edge}'] - min_weight) / (max_weight - min_weight) if max_weight != min_weight else 0.5
      size = min_size + normalized * (max_size - min_size)
      fontsize = min_font + normalized * (max_font - min_font)
      lowest_rank.node(str(node['id']), label = node['token'], color = '#363636', style = 'filled', fillcolor = node_colors[i], shape = 'oval', width = str(1.85 * size), height = str(size), fixedsize = 'true', fontsize = str(fontsize))

  for i, row in graph_df.iterrows():
    graph.edge(str(row['from_index']), str(row['to_index']), arrowsize = '0.5', color = edge_colors[i])
  os.makedirs(os.path.join(STORAGE_PATH, method), exist_ok = True)
  graph.render(directory = os.path.join(STORAGE_PATH, method, f'{filename.replace(".pt", "")}-{edge}'), format = 'png', view = True)

In [ ]:
for i in [16, 17, 18, 92, 143, 224, 338]:
  visualize_graph(path = './graphs/', method = 'chefer_importance', filename = f'SST-2-test-{i}.pt', edge = 0)
  visualize_graph(path = './graphs/', method = 'chefer_importance', filename = f'SST-2-test-{i}.pt', edge = 1)
# for i in [1, 11, 13, 135, 143, 178, 279]:
#   visualize_graph(path = './graphs/', method = 'chefer_importance', filename = f'IMDb-1k-test-{i}.pt', edge = 0)
#   visualize_graph(path = './graphs/', method = 'chefer_importance', filename = f'IMDb-1k-test-{i}.pt', edge = 1)

In [6]:
for i in [1, 11, 13, 135, 143, 178, 279]:
  visualize_graph(path = './graphs/', method = 'chefer_importance', filename = f'IMDb-1k-test-{i}.pt', edge = 0)
  visualize_graph(path = './graphs/', method = 'chefer_importance', filename = f'IMDb-1k-test-{i}.pt', edge = 1)

dot: graph is too large for cairo-renderer bitmaps. Scaling by 0.633558 to fit
dot: graph is too large for cairo-renderer bitmaps. Scaling by 0.616176 to fit
dot: graph is too large for cairo-renderer bitmaps. Scaling by 0.55805 to fit
dot: graph is too large for cairo-renderer bitmaps. Scaling by 0.57164 to fit
dot: graph is too large for cairo-renderer bitmaps. Scaling by 0.339108 to fit
dot: graph is too large for cairo-renderer bitmaps. Scaling by 0.338737 to fit
dot: graph is too large for cairo-renderer bitmaps. Scaling by 0.485891 to fit
dot: graph is too large for cairo-renderer bitmaps. Scaling by 0.516618 to fit
dot: graph is too large for cairo-renderer bitmaps. Scaling by 0.536786 to fit
dot: graph is too large for cairo-renderer bitmaps. Scaling by 0.540389 to fit
dot: graph is too large for cairo-renderer bitmaps. Scaling by 0.577606 to fit
dot: graph is too large for cairo-renderer bitmaps. Scaling by 0.619848 to fit
dot: graph is too large for cairo-renderer bitmaps. Sc

In [7]:
for i in [16, 17, 18, 92, 143, 224, 338]:
  visualize_graph(path = './graphs/', method = 'chefer_importance-looser_threshold', filename = f'SST-2-test-{i}.pt', edge = 0)
  visualize_graph(path = './graphs/', method = 'chefer_importance-looser_threshold', filename = f'SST-2-test-{i}.pt', edge = 1)
# for i in [1, 11, 13, 135, 143, 178, 279]:
#   visualize_graph(path = './graphs/', method = 'chefer_importance', filename = f'IMDb-1k-test-{i}.pt', edge = 0)
#   visualize_graph(path = './graphs/', method = 'chefer_importance', filename = f'IMDb-1k-test-{i}.pt', edge = 1)

In [2]:
from io import StringIO
data = '''0.3236261,0.6763739,338,test
0.50850594,0.4914941,338,test
0.35002074,0.6499793,338,test
0.23163739,0.7683626,338,test
0.29603577,0.70396423,338,test
0.2669731,0.73302686,338,test
0.19790864,0.8020913,338,test
0.48516518,0.5148348,338,test
0.24295723,0.75704277,338,test
0.2556474,0.7443526,338,test'''
sst_2_338_probabilities_df = pd.read_csv(StringIO(data), header = None, names = ['0', '1', 'identifier', 'split'])

In [9]:
(sst_2_338_probabilities_df["1"] >= 0.5).sum()

9

In [8]:
f'{sst_2_338_probabilities_df["1"].mean()} ± {sst_2_338_probabilities_df["1"].std()}'

'0.6841522459999999 ± 0.10526855051013322'

In [7]:
f'{sst_2_338_probabilities_df[sst_2_338_probabilities_df["1"] > 0.6]["1"].mean()} ± {sst_2_338_probabilities_df[sst_2_338_probabilities_df["1"] > 0.6]["1"].std()}'

'0.729399195 ± 0.050098987226648455'